# Build monthly churn train dataset from 2017-01 onward

Notebook này tái sử dụng pipeline hiện tại của dự án để tạo bộ train theo **grain = 1 user / 1 snapshot_month**.

Nguồn logic chính:
- `apps.batch.precompute_tab1_history`: build feature snapshot theo tháng
- `apps/api_fastapi.main` và `apps/dashboard_streamlit.app`: các metric dùng cho predictive layer

Nhãn huấn luyện được dựng là `label_next_month_churn`: user có churn ở **tháng kế tiếp** hay không.

Lưu ý: `train_clean.csv` hiện không có mốc thời gian theo tháng, nên notebook này không dùng file đó để dựng nhãn monthly.

In [ ]:
from __future__ import annotations

import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'apps').exists() and (candidate / 'data').exists():
            return candidate
    raise RuntimeError('Cannot find project root containing apps/ and data/.')


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from apps.batch.precompute_tab1_history import (
    _build_snapshot_rows,
    _load_log_features,
    _load_members,
    _load_tx_features,
)
from apps.producers.common.config import get_settings

settings = get_settings()
PROJECT_ROOT

In [ ]:
START_MONTH = date(2017, 1, 1)
SNAPSHOT_END_MONTH = None  # None => tự lấy tháng cuối hiện có từ user_logs_clean.csv
CHUNK_SIZE = 500_000
OUTPUT_CSV = settings.processed_data_dir / 'train_dataset_monthly_2017_01_onward.csv'


def add_months(month_start: date, delta_months: int) -> date:
    month_index = month_start.year * 12 + (month_start.month - 1) + delta_months
    year = month_index // 12
    month = month_index % 12 + 1
    return date(year, month, 1)


def scan_max_month(csv_path: Path, date_col: str, chunksize: int = 1_000_000) -> date:
    max_ts = None
    for chunk in pd.read_csv(csv_path, usecols=[date_col], chunksize=chunksize):
        values = pd.to_datetime(chunk[date_col], errors='coerce')
        chunk_max = values.max()
        if pd.isna(chunk_max):
            continue
        if max_ts is None or chunk_max > max_ts:
            max_ts = chunk_max
    if max_ts is None:
        raise ValueError(f'Cannot infer max month from {csv_path}')
    return max_ts.to_period('M').to_timestamp().date()


def resolve_snapshot_end_month(explicit_month: date | None) -> date:
    if explicit_month is not None:
        return explicit_month
    return scan_max_month(settings.user_logs_clean_path, 'date')


snapshot_end_month = resolve_snapshot_end_month(SNAPSHOT_END_MONTH)
feature_end_exclusive = add_months(snapshot_end_month, 1)
label_source_end_exclusive = add_months(snapshot_end_month, 2)

print({
    'start_month': START_MONTH.isoformat(),
    'snapshot_end_month': snapshot_end_month.isoformat(),
    'feature_end_exclusive': feature_end_exclusive.isoformat(),
    'label_source_end_exclusive': label_source_end_exclusive.isoformat(),
    'output_csv': str(OUTPUT_CSV),
})

In [ ]:
def load_tx_model_metrics(path: Path, start_month: date, feature_end_exclusive: date) -> pd.DataFrame:
    tx = pd.read_csv(
        path,
        usecols=['msno', 'membership_expire_date', 'actual_amount_paid', 'payment_plan_days'],
        parse_dates=['membership_expire_date'],
    )
    tx = tx.loc[
        (tx['membership_expire_date'] >= pd.Timestamp(start_month))
        & (tx['membership_expire_date'] < pd.Timestamp(feature_end_exclusive))
    ].copy()
    tx['feature_month'] = tx['membership_expire_date'].dt.to_period('M').dt.to_timestamp().dt.date
    tx['actual_amount_paid'] = pd.to_numeric(tx['actual_amount_paid'], errors='coerce').fillna(0.0)
    tx['payment_plan_days'] = pd.to_numeric(tx['payment_plan_days'], errors='coerce').fillna(30.0).clip(lower=1.0)
    return tx.groupby(['msno', 'feature_month'], as_index=False).agg(
        avg_paid=('actual_amount_paid', 'mean'),
        avg_plan_days=('payment_plan_days', 'mean'),
    )


def build_activity_metrics(logs: pd.DataFrame) -> pd.DataFrame:
    if logs.empty:
        return pd.DataFrame(columns=['msno', 'feature_month', 'active_days', 'total_secs', 'avg_daily_secs', 'has_logs_in_month'])

    out = logs[['msno', 'snapshot_month', 'active_days', 'total_secs']].copy()
    out['feature_month'] = out['snapshot_month'].dt.date
    out['active_days'] = pd.to_numeric(out['active_days'], errors='coerce').fillna(0.0)
    out['total_secs'] = pd.to_numeric(out['total_secs'], errors='coerce').fillna(0.0)
    out['avg_daily_secs'] = out['total_secs'].div(out['active_days'].replace(0, pd.NA)).fillna(0.0)
    out['has_logs_in_month'] = (out['active_days'] > 0).astype(int)
    return out[['msno', 'feature_month', 'active_days', 'total_secs', 'avg_daily_secs', 'has_logs_in_month']]


def build_monthly_training_dataset(
    start_month: date,
    snapshot_end_month: date,
    chunk_size: int = 500_000,
) -> pd.DataFrame:
    feature_end_exclusive = add_months(snapshot_end_month, 1)
    label_source_end_exclusive = add_months(snapshot_end_month, 2)

    tx_all = _load_tx_features(
        str(settings.transactions_clean_path),
        start_month,
        label_source_end_exclusive,
    )
    feature_tx = tx_all.loc[
        (tx_all['snapshot_month'] >= pd.Timestamp(start_month))
        & (tx_all['snapshot_month'] < pd.Timestamp(feature_end_exclusive))
    ].copy()
    if feature_tx.empty:
        raise ValueError('No transaction snapshots found in selected window.')

    feature_tx['feature_month'] = feature_tx['snapshot_month'].dt.date
    tx_base = feature_tx[['msno', 'feature_month', 'tx_count', 'avg_price_per_day']].copy()

    candidate_msnos = set(feature_tx['msno'].astype(str).tolist())
    members = _load_members(str(settings.members_clean_path), candidate_msnos, chunk_size)
    logs = _load_log_features(
        str(settings.user_logs_clean_path),
        candidate_msnos,
        start_month,
        feature_end_exclusive,
        chunk_size,
    )

    snapshot_rows = _build_snapshot_rows(feature_tx, members, logs).rename(
        columns={
            'snapshot_month': 'feature_month',
            'churned': 'churned_current_month',
        }
    )

    tx_metrics = load_tx_model_metrics(settings.transactions_clean_path, start_month, feature_end_exclusive)
    activity_metrics = build_activity_metrics(logs)

    label_map = tx_all[['msno', 'snapshot_month', 'churned']].copy()
    label_map['feature_month'] = (label_map['snapshot_month'] - pd.DateOffset(months=1)).dt.date
    label_map = label_map.rename(columns={'churned': 'label_next_month_churn'})
    label_map = label_map[['msno', 'feature_month', 'label_next_month_churn']]

    train_df = (
        snapshot_rows
        .merge(tx_base, on=['msno', 'feature_month'], how='left')
        .merge(tx_metrics, on=['msno', 'feature_month'], how='left')
        .merge(activity_metrics, on=['msno', 'feature_month'], how='left')
        .merge(label_map, on=['msno', 'feature_month'], how='left')
    )

    train_df['label_month'] = train_df['feature_month'].apply(lambda d: add_months(d, 1))
    train_df['avg_paid'] = pd.to_numeric(train_df['avg_paid'], errors='coerce').fillna(0.0)
    train_df['avg_plan_days'] = pd.to_numeric(train_df['avg_plan_days'], errors='coerce').fillna(30.0).clip(lower=1.0)
    train_df['tx_count'] = pd.to_numeric(train_df['tx_count'], errors='coerce').fillna(1).astype(int)
    train_df['avg_price_per_day'] = pd.to_numeric(train_df['avg_price_per_day'], errors='coerce').fillna(0.0)
    train_df['active_days'] = pd.to_numeric(train_df['active_days'], errors='coerce').fillna(0).astype(int)
    train_df['total_secs'] = pd.to_numeric(train_df['total_secs'], errors='coerce').fillna(0.0)
    train_df['avg_daily_secs'] = pd.to_numeric(train_df['avg_daily_secs'], errors='coerce').fillna(0.0)
    train_df['has_logs_in_month'] = pd.to_numeric(train_df['has_logs_in_month'], errors='coerce').fillna(0).astype(int)
    train_df['label_next_month_churn'] = pd.to_numeric(train_df['label_next_month_churn'], errors='coerce').fillna(0).astype(int)

    train_df['manual_renew_flag'] = (1 - train_df['is_auto_renew'].astype(int)).clip(lower=0, upper=1)
    train_df['low_activity_flag'] = (train_df['avg_daily_secs'] < 1800).astype(int)
    train_df['high_skip_flag'] = (train_df['skip_ratio'] > 0.5).astype(int)
    train_df['low_discovery_flag'] = (train_df['discovery_ratio'] < 0.15).astype(int)

    keep_columns = [
        'feature_month',
        'label_month',
        'msno',
        'last_expire_date',
        'churned_current_month',
        'label_next_month_churn',
        'is_auto_renew',
        'manual_renew_flag',
        'survival_days',
        'tx_count',
        'avg_paid',
        'avg_plan_days',
        'avg_price_per_day',
        'active_days',
        'total_secs',
        'avg_daily_secs',
        'discovery_ratio',
        'skip_ratio',
        'low_activity_flag',
        'high_skip_flag',
        'low_discovery_flag',
        'has_logs_in_month',
        'age_bucket',
        'gender_bucket',
        'txn_freq_bucket',
        'skip_ratio_bucket',
        'price_segment',
        'loyalty_segment',
        'active_segment',
    ]

    train_df = train_df[keep_columns].sort_values(['feature_month', 'msno']).reset_index(drop=True)
    return train_df


In [ ]:
train_df = build_monthly_training_dataset(
    start_month=START_MONTH,
    snapshot_end_month=snapshot_end_month,
    chunk_size=CHUNK_SIZE,
)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
train_df.to_csv(OUTPUT_CSV, index=False)

print(f'Rows: {len(train_df):,}')
print(f'Output: {OUTPUT_CSV}')
display(train_df.head(10))

In [ ]:
summary = train_df.groupby('feature_month', as_index=False).agg(
    users=('msno', 'nunique'),
    positive_labels=('label_next_month_churn', 'sum'),
    positive_rate=('label_next_month_churn', 'mean'),
    users_with_logs=('has_logs_in_month', 'sum'),
)
summary['positive_rate'] = summary['positive_rate'].round(6)
display(summary)

print('Overall label rate:', round(train_df['label_next_month_churn'].mean(), 6))
print('Rows without logs:', int((train_df['has_logs_in_month'] == 0).sum()))